<a href="https://colab.research.google.com/github/niya-c-anto/Learnings/blob/main/RAG_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installation of the libraries.

In [ ]:
pip install langchain_community langchainhub chromadb langchain langchain-openai langchain-chroma

Importing the API key

In [ ]:
import os
from google.colab import userdata

# Retrieve the API key from Colab Secrets
# Make sure you have stored your OpenAI API key in Colab Secrets
# with the name 'OPENAI_API_KEY'
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

Web Scrapping

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Corrected: Added a trailing comma to make it a single-element tuple
loader = WebBaseLoader(web_paths=("https://huggingface.co/docs",))
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://huggingface.co/docs', 'title': 'Hugging Face - Documentation', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nHugging Face - Documentation\n\n\n\n\n\n\n\n    Hugging Face        Models  Datasets  Spaces  Buckets new Docs  Enterprise  Pricing     Website  Tasks  HuggingChat  Collections  Languages  Organizations  Community  Blog  Posts  Daily Papers  Hardware  Learn  Discord  Forum  GitHub  Solutions  Team & Enterprise  Hugging Face PRO  Enterprise Support  Inference Providers  Inference Endpoints  Storage Buckets    Log In Sign Up       Documentation Guides, references, and API docs for the Hugging Face ecosystem.     Hub & Client Libraries      Hub Host Git-based models, datasets, and Spaces on the HF Hub     Hub Python Library Python client to interact with the Hugging F

Splitting the docs.

In [ ]:
#from langchain  docs.chunksize=100,chunk overlap=200
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter =RecursiveCharacterTextSplitter(chunk_size = 100,chunk_overlap = 20)
splits = text_splitter.split_documents(docs)

In [ ]:
print(splits[0])
print(splits[1])
print(len(splits))

page_content='Hugging Face - Documentation' metadata={'source': 'https://huggingface.co/docs', 'title': 'Hugging Face - Documentation', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}
page_content='Hugging Face        Models  Datasets  Spaces  Buckets new Docs  Enterprise  Pricing     Website' metadata={'source': 'https://huggingface.co/docs', 'title': 'Hugging Face - Documentation', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}
47


vector DB-chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=splits,embedding=OpenAIEmbeddings())

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
retriever = vectorstore.as_retriever()

NameError: name 'vectorstore' is not defined

In [ ]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

ImportError: cannot import name 'hub' from 'langchain' (/usr/local/lib/python3.12/dist-packages/langchain/__init__.py)

In [ ]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0)

In [ ]:
from langchain_core.runnables import RunnablePasstthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:
def format_docs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [ ]:

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePasstthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke{"When will we get the recordings?"}